# 03 · EVE — an unsupervised evolutionary model

**CFTR variant-effect toolkit · beginner track.** EVE (*Evolutionary model of Variant Effect*, Frazer et al. 2021, *Nature*, PMID 34707284) is a deep generative model of a protein family's evolutionary variation. It is **unsupervised** — it never saw a clinical label — which is what makes it fair to benchmark against ClinVar later (tools/10).

### The EVE data file — where it comes from, and its columns

The build cell below reads `variant_files/CFTR_HUMAN.csv` from the EVE release zip
(evemodel.org; CFTR = UniProt **P13569**). The **zip** also contains the multiple-
sequence alignments (`MSAs/`) and ROC/PRC curve images. `CFTR_HUMAN.csv` has **~42
columns**; the build keeps five, plus one version-tracking column it derives itself:

| column (source) | meaning | in the extract |
|---|---|---|
| `wt_aa`, `position`, `mt_aa` | wild-type AA, residue number, mutant AA | → `protein_variant` (e.g. `M1A`) |
| `EVE_scores_ASM` | EVE pathogenicity, 0–1 (higher = more pathogenic) | → `eve_score` |
| `EVE_classes_75_pct_retained_ASM` | Benign/Pathogenic call at the threshold that leaves the least-confident **25% as "Uncertain"** | → `eve_class` |
| *(derived, not a CSV column)* | the zip's own embedded timestamp for this file (see below) | → `eve_release` |

The other ~37 columns (not used here) include `evolutionary_index_ASM`, `uncertainty_ASM`,
EVE classes at every retention threshold from 10–90%, a separate **BPU** model, and
ClinVar / frequency / ACMG-style flag columns — see the EVE docs.

**Coverage caveat:** EVE is **not** full saturation. It scores only alignable positions
(limited by alignment depth), so the ~26,809 scored variants cover a *subset* of the
1,480 × 19 possible — unlike ESM1b (tools/04), which is saturation.


> ✅ **REAL DATA.** This notebook uses the real EVE scores for CFTR (**~26,809** scored variants, evemodel.org / UniProt P13569), built by the cell below from a manually-downloaded release zip → `data/eve_cftr_2021-08.csv`, and returned by `tk.load_eve()`. Every table's `source` column reads `REAL`. *(License: MIT -- see the fetch cell below for the confirmed source.)*


In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · EVE — an evolutionary generative model

**What is it?** EVE (*Evolutionary model of Variant Effect*) is a deep **generative model** trained on the **multiple-sequence alignment (MSA)** of a protein family — i.e. the same protein lined up across hundreds or thousands of species. Evolution has already run a giant natural experiment: positions that *must not change* to keep the protein working stay constant across species, while positions that tolerate change vary freely.

EVE learns that pattern of *evolutionary constraint*, then scores a new variant by asking: **"how well does this amino-acid change fit what evolution has tolerated here?"** A change at a rigidly-conserved position looks evolutionarily *implausible* → high score.

- **Score range:** `[0, 1]` (a posterior probability of being pathogenic).
- **Rule of thumb:** `>= 0.5` ~ pathogenic, `< 0.5` ~ benign. Higher = more damaging.
- **Unsupervised:** trained only on sequences, *never* on ClinVar labels → **low circularity** when we later benchmark it against clinical databases.

> **EVE resources:** paper Frazer et al. 2021, *Nature*, PMID 34707284; scores at **evemodel.org**; open-source code (MIT) at **github.com/OATML-Markslab/EVE**.

## Building the REAL data — a manual download (no API, no small per-gene file)

EVE ships neither an API nor a per-gene download — the only way to get it is the
full release zip from **evemodel.org**, which bundles every human protein's
scores plus MSAs and ROC/PRC images (~9.6 GB). **You must fetch this one
yourself:**

1. Go to <https://evemodel.org>, register/log in, and download **`EVE_all_data.zip`**
   (release 2021-08).
2. Save it as `data/EVE_all_data.zip` (gitignored — never commit it).

The cell below then does the actual build: it opens the zip **without
extracting it**, reads only the one CFTR member (`variant_files/CFTR_HUMAN.csv`,
~42 columns, of which we keep 5), drops the residues EVE didn't score, and
writes the small extract `data/eve_cftr_2021-08.csv` used by every EVE cell
below. Skips rebuilding if that file already exists.

**License: MIT — confirmed on evemodel.org, not just assumed from the code
repo.** Both the site's "Bulk Protein Data" and "Single Protein Data" download
pages state, verbatim: *"The downloading of this data, and of all other data
on this site, falls under the MIT License."* (checked 2026-08-07). That
explicitly covers the **scores themselves**, not just the GitHub code — MIT
permits use, modification, and redistribution (including commercial), with the
only requirement being to keep the copyright notice and license text attached.
No NonCommercial or ShareAlike restriction, unlike several other tools in this
toolkit (REVEL, PrimateAI, SpliceAI). See `data_manifest.json`.

**Version & reproducibility.** Unlike gnomAD/ClinVar/AlphaMissense, this build
has no live HTTP request to grab a `Last-Modified` header from — the zip is a
manual, authenticated download, not something this cell fetches itself. But the
zip is not unversioned either: every member file inside a zip archive carries
its own embedded timestamp, recorded by whoever packaged it. For
`variant_files/CFTR_HUMAN.csv`, that embedded timestamp is
**2021-07-08 21:00:44** — read straight out of the zip's own metadata
(`zipfile.ZipInfo.date_time`), no download needed. That's EVE's own team's
record of when *that exact CSV* was packaged, which is a more precise signal
than the release's general "2021-08" label (the zip's `PRC_20210822`/
`ROC_20210822` folders show the performance-curve images were generated about
six weeks later). The build cell records it into
`data/eve_cftr_2021-08.release.json`; `load_eve()` exposes it as the
`eve_release` column, so it's visible on every table downstream, the same way
`am_release` is for AlphaMissense.


In [2]:
import zipfile, io, json
from datetime import datetime

DATA_DIR = pathlib.Path.cwd().parent / "data"
EVE_ZIP = DATA_DIR / "EVE_all_data.zip"
EVE_MEMBER = "variant_files/CFTR_HUMAN.csv"
EVE_TSV = DATA_DIR / "eve_cftr_2021-08.csv"
EVE_RELEASE_JSON = DATA_DIR / "eve_cftr_2021-08.release.json"

if EVE_TSV.exists():
    print(f"already built -> {EVE_TSV.name} (delete it and {EVE_RELEASE_JSON.name} to rebuild)")
elif not EVE_ZIP.exists():
    raise FileNotFoundError(
        f"{EVE_ZIP} not found.\n"
        "EVE has no API and no small per-gene download -- get the full release zip:\n"
        "  1. Go to https://evemodel.org, register/log in, and download 'EVE_all_data.zip'\n"
        "     (release 2021-08; the zip also bundles MSAs/ and ROC/PRC images; ~9.6 GB)\n"
        f"  2. Save it as {EVE_ZIP} (do NOT commit it -- data/ is gitignored)\n"
        "Then re-run this cell -- it reads only the one CFTR member file from inside the zip."
    )
else:
    with zipfile.ZipFile(EVE_ZIP) as z:
        member_info = z.getinfo(EVE_MEMBER)
        # The zip's own internal metadata: when EVE's team packaged this exact
        # file, straight from the archive -- no download or HTTP call needed.
        packaged_at = datetime(*member_info.date_time).isoformat()
        with z.open(EVE_MEMBER) as fh:
            raw = pd.read_csv(io.TextIOWrapper(fh, encoding="utf-8"),
                               usecols=["wt_aa", "position", "mt_aa",
                                        "EVE_scores_ASM", "EVE_classes_75_pct_retained_ASM"],
                               low_memory=False)
    print("rows in CFTR_HUMAN.csv:", len(raw))
    print(f"CFTR_HUMAN.csv packaged at (zip-embedded timestamp): {packaged_at}")
    df = raw.dropna(subset=["EVE_scores_ASM"]).copy()   # keep only rows EVE actually scored
    df["protein_variant"] = (df["wt_aa"].astype(str) + df["position"].astype(int).astype(str)
                              + df["mt_aa"].astype(str))
    df = df.rename(columns={"EVE_scores_ASM": "eve_score",
                             "EVE_classes_75_pct_retained_ASM": "eve_class"})
    df["source"] = "REAL"
    out = df[["protein_variant", "wt_aa", "position", "mt_aa", "eve_score", "eve_class", "source"]]
    out = out.sort_values("position").reset_index(drop=True)
    out.to_csv(EVE_TSV, index=False)
    EVE_RELEASE_JSON.write_text(json.dumps({
        "zip_member": EVE_MEMBER,
        "zip_member_packaged_at": packaged_at,
        "release_label": "2021-08",
        "note": "packaged_at is the zip's own embedded per-file timestamp (zipfile.ZipInfo.date_time), "
                "not a download date -- it is EVE's own record of when this exact CSV was built.",
    }, indent=2))
    print(f"scored EVE variants written: {len(out):,} -> {EVE_TSV.relative_to(DATA_DIR.parent)}")
    print(f"wrote {EVE_RELEASE_JSON.relative_to(DATA_DIR.parent)}")


rows in CFTR_HUMAN.csv: 29600
CFTR_HUMAN.csv packaged at (zip-embedded timestamp): 2021-07-08T21:00:44


scored EVE variants written: 26,809 -> data\eve_cftr_2021-08.csv
wrote data\eve_cftr_2021-08.release.json


In [3]:
eve = tk.load_eve()      # REAL — CFTR EVE extract (built by the cell above from evemodel.org)
print(f'{len(eve)} REAL EVE variants | source: {eve["source"].unique().tolist()}')
print(f'residues covered: {eve["protein_variant"].str.extract(r"(\d+)")[0].nunique()}')
print(f'eve_release (zip-embedded timestamp): {eve["eve_release"].iloc[0]}')
print()
print(eve['eve_class'].value_counts().to_string())
eve.head(8)


26809 REAL EVE variants | source: ['REAL']
residues covered: 1411
eve_release (zip-embedded timestamp): 2021-07-08T21:00:44

eve_class
Pathogenic    15879
Uncertain      5536
Benign         5394


,protein_variant,eve_score,eve_class,eve_release,source
0,L15A,0.520237,Uncertain,2021-07-08T21:00:44,REAL
1,L15C,0.600187,Uncertain,2021-07-08T21:00:44,REAL
2,L15D,0.831544,Pathogenic,2021-07-08T21:00:44,REAL
3,L15E,0.579939,Uncertain,2021-07-08T21:00:44,REAL
4,L15F,0.237468,Benign,2021-07-08T21:00:44,REAL
5,L15G,0.625586,Uncertain,2021-07-08T21:00:44,REAL
6,L15H,0.625182,Uncertain,2021-07-08T21:00:44,REAL
7,L15I,0.177613,Benign,2021-07-08T21:00:44,REAL


## 2 · Turn an EVE score into a call

`tk.call_from_score(score, 'eve')` applies EVE's published threshold (`>= 0.5` ~ pathogenic, higher = worse) and returns `pathogenic / uncertain / benign`.

In [4]:
eve['eve_call'] = eve['eve_score'].apply(lambda s: tk.call_from_score(s, 'eve'))
eve[['protein_variant', 'eve_score', 'eve_class', 'eve_call', 'source']].head(12)

,protein_variant,eve_score,eve_class,eve_call,source
0,L15A,0.520237,Uncertain,pathogenic,REAL
1,L15C,0.600187,Uncertain,pathogenic,REAL
2,L15D,0.831544,Pathogenic,pathogenic,REAL
3,L15E,0.579939,Uncertain,pathogenic,REAL
4,L15F,0.237468,Benign,benign,REAL
5,L15G,0.625586,Uncertain,pathogenic,REAL
6,L15H,0.625182,Uncertain,pathogenic,REAL
7,L15I,0.177613,Benign,benign,REAL
8,L15K,0.599925,Uncertain,pathogenic,REAL
9,L15M,0.201823,Benign,benign,REAL


## Example: the shared missense worked-example panel, scored by **EVE**

The same fixed panel of famous CFTR **missense** variants runs through every missense tool
(tools/01–06, benchmark/00–01), so you can follow one set of variants across the series. The
variant list is `tk.A1_PANEL_VARIANTS` / `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the
**scoring is shown inline below** so you can see exactly how EVE is joined onto it.

In [5]:
panel = tk.A1_PANEL_VARIANTS
eve = tk.load_eve()
eve[eve['protein_variant'].isin(panel)][['protein_variant', 'eve_score', 'eve_class']].reset_index(drop=True)

,protein_variant,eve_score,eve_class
0,G85E,0.922779,Pathogenic
1,R117H,0.819590,Pathogenic
2,Y161C,0.934065,Pathogenic
3,P205S,0.943775,Pathogenic
4,R334W,0.950636,Pathogenic
5,V520F,0.948129,Pathogenic
6,G551D,0.939098,Pathogenic
7,R668C,0.921812,Pathogenic
8,S912L,0.085494,Benign
9,H949Y,0.804175,Pathogenic


## Key takeaways

1. **EVE** models a protein family's **evolutionary** constraint (its MSA). Score `[0,1]`, `>= 0.5` ~ pathogenic, **higher = worse**.
2. **Unsupervised** w.r.t. clinical labels → **low circularity** vs ClinVar.
3. This notebook uses **REAL EVE** — ~26,809 CFTR variants (evemodel.org / P13569, MIT-licensed), shipped as `data/eve_cftr_2021-08.csv`; `source` reads `REAL`.
4. **Version-tracked despite the manual download**: the build cell reads the release zip's own embedded per-file timestamp (`eve_release` = 2021-07-08T21:00:44 for CFTR_HUMAN.csv) rather than trusting the general "2021-08" release label.

**Next:** tools/04 — **ESM1b**, a protein language model whose scale runs *backwards*. The EVE-vs-ESM1b comparison lives in the archived integration notebook.
